<a href="https://colab.research.google.com/github/sannyskystar/ai-credit-risk-fairness-analyzer/blob/main/AI_Credit_Risk_Fairness_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Loan Approval Bias & Fairness Analyzer

A research-based machine learning project that:
- Predicts credit risk
- Measures model fairness across demographic groups
- Applies bias mitigation
- Compares fairness before vs after mitigation

In [1]:
from google.colab import files

uploaded = files.upload()

Saving german_credit_data.csv to german_credit_data (1).csv


In [2]:
import pandas as pd

print(uploaded.keys())

dict_keys(['german_credit_data (1).csv'])


In [3]:
import pandas as pd

file_name = "german_credit_data (1).csv"

df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (1000, 10)


,Unnamed: 0,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
0,0,67,male,2,own,NaN,little,1169,6,radio/TV
1,1,22,female,2,own,little,moderate,5951,48,radio/TV
2,2,49,male,1,own,little,NaN,2096,12,education
3,3,45,male,2,free,little,little,7882,42,furniture/equipment
4,4,53,male,2,free,little,little,4870,24,car


In [4]:
from google.colab import files

uploaded = files.upload()

Saving german_credit_data (1)-target.csv to german_credit_data (1)-target.csv


In [5]:
import pandas as pd

file_name = "german_credit_data (1)-target.csv"

df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (1000, 11)


,Unnamed: 0,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,0,67,male,2,own,NaN,little,1169,6,radio/TV,good
1,1,22,female,2,own,little,moderate,5951,48,radio/TV,bad
2,2,49,male,1,own,little,NaN,2096,12,education,good
3,3,45,male,2,free,little,little,7882,42,furniture/equipment,good
4,4,53,male,2,free,little,little,4870,24,car,bad


In [6]:
print("Column names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df["Risk"].value_counts())

print("\nSex distribution:")
print(df["Sex"].value_counts())

Column names:
['Unnamed: 0', 'Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Purpose', 'Risk']

Missing values:
Unnamed: 0            0
Age                   0
Sex                   0
Job                   0
Housing               0
Saving accounts     183
Checking account    394
Credit amount         0
Duration              0
Purpose               0
Risk                  0
dtype: int64

Target distribution:
Risk
good    700
bad     300
Name: count, dtype: int64

Sex distribution:
Sex
male      690
female    310
Name: count, dtype: int64


In [7]:
# Remove the unnecessary index column
df = df.drop(columns=["Unnamed: 0"])

print("Remaining columns:")
print(df.columns.tolist())

Remaining columns:
['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Purpose', 'Risk']


In [8]:
# Fill missing categorical values
df["Saving accounts"] = df["Saving accounts"].fillna("unknown")
df["Checking account"] = df["Checking account"].fillna("unknown")

# Check that there are no missing values left
print(df.isnull().sum())

Age                 0
Sex                 0
Job                 0
Housing             0
Saving accounts     0
Checking account    0
Credit amount       0
Duration            0
Purpose             0
Risk                0
dtype: int64


In [9]:
# Separate features and target
X = df.drop(columns=["Risk"])
y = df["Risk"]

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

Features:
['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Purpose']

Target:
Risk


In [10]:
# Convert target labels to numbers
y = y.map({"bad": 0, "good": 1})

print(y.value_counts())
print("\nUnique target values:", y.unique())

Risk
1    700
0    300
Name: count, dtype: int64

Unique target values: [1 0]


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Identify categorical columns
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

# Create the encoder
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_columns)
    ],
    remainder="passthrough"
)

print("\nPreprocessor created successfully!")

Categorical columns:
['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']

Preprocessor created successfully!


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training samples: 800
Testing samples: 200

Training target distribution:
Risk
1    560
0    240
Name: count, dtype: int64

Testing target distribution:
Risk
1    140
0     60
Name: count, dtype: int64


In [13]:
# Fit the preprocessor on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data using the same preprocessor
X_test_processed = preprocessor.transform(X_test)

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("Original test shape:", X_test.shape)
print("Processed test shape:", X_test_processed.shape)

Original training shape: (800, 9)
Processed training shape: (800, 26)
Original test shape: (200, 9)
Processed test shape: (200, 26)


In [14]:
from sklearn.linear_model import LogisticRegression

# Create the model
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train the model
model.fit(X_train_processed, y_train)

print("Model trained successfully!")

Model trained successfully!


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Scale the processed features
scaler = StandardScaler(with_mean=False)

X_train_scaled = scaler.fit_transform(X_train_processed)
X_test_scaled = scaler.transform(X_test_processed)

# Create the model
model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

# Train
model.fit(X_train_scaled, y_train)

print("Model trained successfully!")

Model trained successfully!


In [16]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Make predictions on unseen test data
y_pred = model.predict(X_test_scaled)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 73.50%

Classification Report:
              precision    recall  f1-score   support

           0       0.57      0.45      0.50        60
           1       0.78      0.86      0.82       140

    accuracy                           0.73       200
   macro avg       0.68      0.65      0.66       200
weighted avg       0.72      0.73      0.72       200


Confusion Matrix:
[[ 27  33]
 [ 20 120]]


In [17]:
# Create a copy of the test data
results = X_test.copy()

# Add actual and predicted outcomes
results["actual"] = y_test.values
results["predicted"] = y_pred

print(results[["Sex", "actual", "predicted"]].head(10))

        Sex  actual  predicted
977    male       1          1
735  female       1          0
615    male       1          1
413    male       1          1
563    male       0          1
27   female       1          1
514    male       1          1
624    male       0          1
475  female       0          0
991    male       1          1


In [18]:
# Prediction rate for each gender
prediction_rates = results.groupby("Sex")["predicted"].mean()

print("Predicted 'Good Credit' rate by gender:")
print(prediction_rates)

print("\nPercentage:")
print((prediction_rates * 100).round(2))

Predicted 'Good Credit' rate by gender:
Sex
female    0.600000
male      0.835714
Name: predicted, dtype: float64

Percentage:
Sex
female    60.00
male      83.57
Name: predicted, dtype: float64


In [19]:
comparison = results.groupby("Sex")[["actual", "predicted"]].mean() * 100

print(comparison.round(2))

        actual  predicted
Sex                      
female   66.67      60.00
male     71.43      83.57


In [20]:
female_rate = results[results["Sex"] == "female"]["predicted"].mean()
male_rate = results[results["Sex"] == "male"]["predicted"].mean()

spd = female_rate - male_rate

print(f"Female predicted-good rate: {female_rate:.2%}")
print(f"Male predicted-good rate:   {male_rate:.2%}")
print(f"Statistical Parity Difference: {spd:.4f}")

Female predicted-good rate: 60.00%
Male predicted-good rate:   83.57%
Statistical Parity Difference: -0.2357


In [21]:
# Calculate Disparate Impact
disparate_impact = female_rate / male_rate

print(f"Disparate Impact: {disparate_impact:.4f}")

Disparate Impact: 0.7179


In [22]:
# Calculate accuracy separately for each gender

for gender in results["Sex"].unique():
    group = results[results["Sex"] == gender]

    accuracy = (group["actual"] == group["predicted"]).mean()

    print(f"{gender.capitalize()} accuracy: {accuracy:.2%}")

Male accuracy: 75.00%
Female accuracy: 70.00%


In [23]:
from sklearn.metrics import recall_score

for gender in results["Sex"].unique():
    group = results[results["Sex"] == gender]

    tpr = recall_score(group["actual"], group["predicted"], pos_label=1)

    print(f"{gender.capitalize()} True Positive Rate: {tpr:.2%}")

Male True Positive Rate: 91.00%
Female True Positive Rate: 72.50%


In [24]:
from sklearn.metrics import confusion_matrix

for gender in results["Sex"].unique():
    group = results[results["Sex"] == gender]

    tn, fp, fn, tp = confusion_matrix(
        group["actual"],
        group["predicted"],
        labels=[0, 1]
    ).ravel()

    fpr = fp / (fp + tn)

    print(f"{gender.capitalize()} False Positive Rate: {fpr:.2%}")

Male False Positive Rate: 65.00%
Female False Positive Rate: 35.00%


In [25]:
fairness_report = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Predicted Good Rate",
        "True Positive Rate",
        "False Positive Rate"
    ],
    "Female": [
        0.70,
        female_rate,
        0.725,
        0.35
    ],
    "Male": [
        0.75,
        male_rate,
        0.91,
        0.65
    ]
})

fairness_report["Difference (Female - Male)"] = (
    fairness_report["Female"] - fairness_report["Male"]
)

print(fairness_report.round(4))

                Metric  Female    Male  Difference (Female - Male)
0             Accuracy   0.700  0.7500                     -0.0500
1  Predicted Good Rate   0.600  0.8357                     -0.2357
2   True Positive Rate   0.725  0.9100                     -0.1850
3  False Positive Rate   0.350  0.6500                     -0.3000


In [26]:
!pip install -q aif360

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 7.4 MB/s eta 0:00:00


In [27]:
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing

print("AIF360 imported successfully!")

pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'


AIF360 imported successfully!


In [28]:
from aif360.datasets import BinaryLabelDataset

# Prepare a simple dataframe for AIF360
train_fairness_df = X_train.copy()

# Add the target
train_fairness_df["Risk"] = y_train.values

# Convert categorical columns to numerical values
train_fairness_df = pd.get_dummies(
    train_fairness_df,
    columns=categorical_columns,
    dtype=int
)

# Create AIF360 dataset
train_dataset = BinaryLabelDataset(
    df=train_fairness_df,
    label_names=["Risk"],
    protected_attribute_names=["Sex_male"]
)

print("AIF360 dataset created successfully!")
print("Shape:", train_dataset.features.shape)

AIF360 dataset created successfully!
Shape: (800, 26)


In [29]:
from aif360.algorithms.preprocessing import Reweighing

# Define privileged and unprivileged groups
# Sex_male = 1 → Male (privileged)
# Sex_male = 0 → Female (unprivileged)

RW = Reweighing(
    unprivileged_groups=[{"Sex_male": 0}],
    privileged_groups=[{"Sex_male": 1}]
)

# Calculate sample weights
train_dataset_reweighted = RW.fit_transform(train_dataset)

print("Reweighing completed successfully!")

print("\nOriginal weights (first 10):")
print(train_dataset.instance_weights[:10])

print("\nNew weights (first 10):")
print(train_dataset_reweighted.instance_weights[:10])

Reweighing completed successfully!

Original weights (first 10):
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

New weights (first 10):
[1.08695652 0.96491228 1.08695652 0.96491228 1.08695652 1.09271523
 0.96491228 1.08695652 0.96491228 0.96491228]


In [30]:
# Create a new Logistic Regression model
fair_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

# Train using the reweighted samples
fair_model.fit(
    X_train_scaled,
    y_train,
    sample_weight=train_dataset_reweighted.instance_weights
)

print("Fairness-mitigated model trained successfully!")

Fairness-mitigated model trained successfully!


In [31]:
# Predictions from the fairness-mitigated model
fair_y_pred = fair_model.predict(X_test_scaled)

# Basic performance
fair_accuracy = accuracy_score(y_test, fair_y_pred)

print(f"Fairness-mitigated model accuracy: {fair_accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(y_test, fair_y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, fair_y_pred))

Fairness-mitigated model accuracy: 73.00%

Classification Report:
              precision    recall  f1-score   support

           0       0.57      0.43      0.49        60
           1       0.78      0.86      0.82       140

    accuracy                           0.73       200
   macro avg       0.67      0.65      0.65       200
weighted avg       0.72      0.73      0.72       200


Confusion Matrix:
[[ 26  34]
 [ 20 120]]


In [32]:
# Add mitigated predictions to our results
results["fair_predicted"] = fair_y_pred

# Calculate fairness metrics for the mitigated model
for gender in ["female", "male"]:
    group = results[results["Sex"] == gender]

    accuracy = (group["actual"] == group["fair_predicted"]).mean()
    predicted_good = group["fair_predicted"].mean()
    tpr = recall_score(group["actual"], group["fair_predicted"], pos_label=1)

    tn, fp, fn, tp = confusion_matrix(
        group["actual"],
        group["fair_predicted"],
        labels=[0, 1]
    ).ravel()

    fpr = fp / (fp + tn)

    print(f"\n{gender.upper()}")
    print(f"Accuracy: {accuracy:.2%}")
    print(f"Predicted Good Rate: {predicted_good:.2%}")
    print(f"True Positive Rate: {tpr:.2%}")
    print(f"False Positive Rate: {fpr:.2%}")


FEMALE
Accuracy: 68.33%
Predicted Good Rate: 71.67%
True Positive Rate: 80.00%
False Positive Rate: 55.00%

MALE
Accuracy: 75.00%
Predicted Good Rate: 79.29%
True Positive Rate: 88.00%
False Positive Rate: 57.50%


In [33]:
female_fair_rate = results[results["Sex"] == "female"]["fair_predicted"].mean()
male_fair_rate = results[results["Sex"] == "male"]["fair_predicted"].mean()

fair_spd = female_fair_rate - male_fair_rate
fair_disparate_impact = female_fair_rate / male_fair_rate

print(f"Fairness-mitigated SPD: {fair_spd:.4f}")
print(f"Fairness-mitigated Disparate Impact: {fair_disparate_impact:.4f}")

Fairness-mitigated SPD: -0.0762
Fairness-mitigated Disparate Impact: 0.9039


In [34]:
import joblib
import os

# Create a folder for our trained models
os.makedirs("models", exist_ok=True)

# Save the models and preprocessing objects
joblib.dump(model, "models/baseline_model.pkl")
joblib.dump(fair_model, "models/fairness_model.pkl")
joblib.dump(preprocessor, "models/preprocessor.pkl")
joblib.dump(scaler, "models/scaler.pkl")

print("All model files saved successfully!")

All model files saved successfully!


In [35]:
import os

print(os.listdir("models"))

['scaler.pkl', 'preprocessor.pkl', 'baseline_model.pkl', 'fairness_model.pkl']


In [36]:
from google.colab import files
import shutil

shutil.make_archive("fairness_models", "zip", "models")

files.download("fairness_models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import sys
import sklearn
import joblib

print("Python:", sys.version)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
scikit-learn: 1.6.1
joblib: 1.5.3
